In [ ]:
import pandas as pd
par = pd.read_parquet("../data/processed/paradas_limpo.parquet")
##código para encontrar os eventos que mais pararam as máquinas MC 68 e MC 63
maq = par[par["Estação"].str.startswith(("MC 68", "MC 63"))]
eventos = maq.groupby("Evento")["dur_min"].agg(["count", "sum"]).sort_values("sum", ascending=False)
eventos["horas"] = eventos["sum"] / 60
print(len(eventos), "eventos diferentes")
eventos

31 eventos diferentes


,count,sum,horas
Evento,,,
Setup Corte,3233,28278.366667,471.306111
Parada Programada para Lanche,1448,24699.450000,411.657500
Reabastecimento de condutor,10429,24110.966667,401.849444
Ajuste Máquina/Equipamento,3381,20620.766667,343.679444
Parada programada para o almoço,322,19069.950000,317.832500
Parada desconhecida,19634,18000.400000,300.006667
Parada não Categorizada,3986,17704.850000,295.080833
Plano de Controle,3893,16164.850000,269.414167
Manutenção MC,672,15599.083333,259.984722


In [8]:
##alguams paradas apareceram duplicadas, provavelmente por um erro já conhecido
#  de o export do sistema MES as vezes gerar algumas paradas com espaço no final do nome do evento.
[repr(e) for e in eventos.index if "almoço" in e or "Autônoma" in e]

["'Parada programada para o almoço'",
 "'Parada programada para o almoço '",
 "'Parada para Manutenção Autônoma'",
 "'Parada para Manutenção Autônoma '"]

In [11]:
#Confirmado que esse era o problema, corrigindo.

import pandas as pd
par = pd.read_parquet("../data/processed/paradas_limpo.parquet")
par["Evento"] = (par["Evento"]
                 .str.strip()                            # tira espaços do começo e do fim
                 .str.replace(r"\s+", " ", regex=True))  # troca espaços repetidos/especiais por um espaço normal
##código para encontrar os eventos que mais pararam as máquinas MC 68 e MC 63
maq = par[par["Estação"].str.startswith(("MC 68", "MC 63"))]
eventos = maq.groupby("Evento")["dur_min"].agg(["count", "sum"]).sort_values("sum", ascending=False)
eventos["horas"] = eventos["sum"] / 60
print(len(eventos), "eventos diferentes")
eventos



29 eventos diferentes


,count,sum,horas
Evento,,,
Setup Corte,3233,28278.366667,471.306111
Parada programada para o almoço,443,26055.000000,434.250000
Parada Programada para Lanche,1448,24699.450000,411.657500
Reabastecimento de condutor,10429,24110.966667,401.849444
Ajuste Máquina/Equipamento,3381,20620.766667,343.679444
Parada desconhecida,19634,18000.400000,300.006667
Parada não Categorizada,3986,17704.850000,295.080833
Plano de Controle,3893,16164.850000,269.414167
Manutenção MC,672,15599.083333,259.984722


In [21]:
par["Evento"] = par["Evento"].str.strip().str.replace(r"\s+", " ", regex=True)

#classificação de familias de eventos de acordo com a consultoria.
def familia(evento):
    e = evento.lower()

    if "não categorizada" in e or "desconhecida" in e: 
        return "sem_categoria" #nao categorizada ou desconhecida
    
    if any(p in e for p in ["programada", "lanche", "planejada", "autônoma", "treinamento"]):
        return "programada" #parada programada
    
    if "setup" in e or "buscar ma" in e:
        return "setup"
    
    if "abastecimento" in e:          # todos os tipos de falta de material ou reabastecimento de material
        return "abastecimento"
    
    if "ajuste" in e or "manutenção" in e: #ajuste operacional ou necessidade de manutenção corretiva em máquina ou mini-aplicador
        return "ajuste_manutencao"          # server ppara os times de manutenção de MA e manutenção mecânica ou elétrica
    
    if any(p in e for p in ["plano de controle", "embramada", "produto novo", "cross"]):  #paradas de qualidade
        return "processo_qualidade"
    # 7º: pessoas e organização
    if any(p in e for p in ["organização", "saída do posto", "ambulatório"]): #organização do posto de trabalho
        return "organizacao_posto"
    # 8º: fora do controle da máquina e do operador
    if any(p in e for p in ["falta de programação", "ti", "sistema", "ar comprimido"]): #problemas externos à fabrica
        return "externo"
    return "outros" #caso haja alguma criação de novo envento, classifica como outros, para que seja analisado posteriormente e classificado corretamente.

par["familia"] = par["Evento"].map(familia)

maq = par[par["Estação"].str.startswith(("MC 68", "MC 63"))]

# Horas paradas por família
print("Horas paradas por família:")
print(maq.groupby("familia")["dur_min"].sum().div(60).sort_values(ascending=False).round(0))

print("eventos em outros: ",maq[maq["familia"] == "outros"]["Evento"].value_counts())

Horas paradas por família:
familia
programada            929.0
ajuste_manutencao     763.0
sem_categoria         595.0
abastecimento         586.0
setup                 476.0
processo_qualidade    418.0
organizacao_posto     258.0
externo                29.0
Name: dur_min, dtype: float64
eventos em outros:  Series([], Name: count, dtype: int64)


In [22]:
ses = pd.read_csv("../data/processed/escp771_sessoes_corte.csv",
                  parse_dates=["ini", "fim"], low_memory=False)
ordens = pd.read_csv("../data/processed/escp771_ordens_reconstruidas.csv", low_memory=False)

#só as máquians de corte
ses = ses[(ses["mc_realizado"].astype(str).isin(["68", "63"])) & (ses["relatorio"] == "corte")]

# junta a receita do item a ser cortado
receita = ["cod_prog", "sub_item", "quantidade", "tempo_prog_min", "condutor",
           "terminal_a", "terminal_b", "ma_lado_a", "ma_lado_b", "comprimento", "cor_cabo"]
ses = ses.merge(ordens[receita], on=["cod_prog", "sub_item"], how="left")

print(ses.shape)
ses.groupby("mc_realizado").size()

(4194, 37)


mc_realizado
63    2139
68    2055
dtype: int64

In [23]:
import numpy as np

# 1) Paradas das duas máquinas, com o fim calculado (início + duração)
p_all = par.copy()
p_all["mc"] = p_all["Estação"].str.extract(r"MC\s*(\d+)")[0]
p_all = p_all[p_all["mc"].isin(["68", "63"])]
p_all["fim_par"] = p_all["Data inicial"] + pd.to_timedelta(p_all["dur_min"], unit="min")

#Sessões válidas, desconsiderando sessões com mais de 24 horas que as vezes ficam abertas aos domingos
ses = pd.read_csv("../data/processed/escp771_sessoes_corte.csv", parse_dates=["ini", "fim"], low_memory=False)
ses = ses[(ses["mc_realizado"].astype(str).isin(["68", "63"])) & (ses["relatorio"] == "corte")].copy()
ses["mc"] = ses["mc_realizado"].astype(str)
ses = ses[ses["fim"].notna() & (ses["dur_util_min"] >= 1) & ~ses["dur_mais_24h"]].reset_index(drop=True)

#Cruzamento por intervalo de tempo
FAMILIAS = ["setup", "abastecimento", "ajuste_manutencao", "processo_qualidade",
            "organizacao_posto", "sem_categoria", "externo", "programada"]
linhas = []
for mc, s_mc in ses.groupby("mc"):
    p = p_all[p_all["mc"] == mc]
    p_ini, p_fim, p_fam = p["Data inicial"].values, p["fim_par"].values, p["familia"].values
    for idx, s in s_mc.iterrows():
        s_ini, s_fim = np.datetime64(s["ini"]), np.datetime64(s["fim"])
        m = (p_ini < s_fim) & (p_fim > s_ini)                       # paradas que se sobrepõem
        dentro = (np.minimum(p_fim[m], s_fim) - np.maximum(p_ini[m], s_ini)) / np.timedelta64(1, "m")
        fam = p_fam[m]
        r = {"idx": idx}
        for f in FAMILIAS:
            r[f"par_{f}_min"] = dentro[fam == f].sum()             # minutos de cada família
        perda = fam != "programada"
        r["n_paradas_perda"] = int(perda.sum())
        r["maior_parada_perda_min"] = dentro[perda].max() if perda.any() else 0.0
        linhas.append(r)
ses = ses.join(pd.DataFrame(linhas).set_index("idx"))

# 4) Alvos
ses["y_abastecimento"] = (ses["par_abastecimento_min"] > 0).astype(int)
ses["y_setup"]         = (ses["par_setup_min"] > 0).astype(int)
ses["y_parada_longa"]  = (ses["maior_parada_perda_min"] >= 15).astype(int)

print(len(ses), "sessões")
ses.groupby("mc")[["y_abastecimento", "y_setup", "y_parada_longa"]].mean().round(3)

4170 sessões


,y_abastecimento,y_setup,y_parada_longa
mc,,,
63,0.712,0.369,0.104
68,0.702,0.419,0.078


In [24]:
# Variáveis conhecidas ANTES do início da sessão (sem vazamento)
X_COLS = [
    "comprimento", "qtd_sessao", "tempo_prog_sessao_min", "quantidade_ordem",
    "troca_condutor", "troca_terminal_a", "troca_terminal_b", "troca_cor", "troca_ma_a", "troca_ma_b",
    "mesmo_sub_item", "mesma_programacao", "ocioso_antes_min", "lacuna_antes_6h",
    "hora_ini", "dia_semana", "ordem_no_dia", "n_subitens_kit",
    "n_ocorrencias_previas_sub", "dur_media_hist_sub", "razao_media_hist_sub",
    "setup_pcp", "maquina_diferente_programada", "flag_registro_suspeito",
]

v4 = pd.concat([
    pd.read_csv(f"../data/processed/dataset_mc{m}_v4_horaria_anual.csv", parse_dates=["ini"], low_memory=False).assign(mc=m)
    for m in ["68", "63"]
])

# Padroniza as chaves para a junção bater
for df in (ses, v4):
    df["cod_prog"] = pd.to_numeric(df["cod_prog"], errors="coerce").astype("Int64")
    df["sub_item"] = df["sub_item"].astype(str)

chave = ["mc", "cod_prog", "sub_item", "ini"]
base = ses.merge(v4[chave + X_COLS], on=chave, how="left")

# Remove os registros que você marcou como suspeitos na revisão manual
base = base[~base["flag_registro_suspeito"].astype(bool)].drop(columns="flag_registro_suspeito")

print(base.shape)
base.to_parquet("../data/processed/dataset_modelagem.parquet")

(4055, 65)


C:\Users\davif\AppData\Local\Temp\ipykernel_20660\4043805703.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pd.read_csv(f"../data/processed/dataset_mc{m}_v4_horaria_anual.csv", parse_dates=["ini"], low_memory=False).assign(mc=m)
C:\Users\davif\AppData\Local\Temp\ipykernel_20660\4043805703.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pd.read_csv(f"../data/processed/dataset_mc{m}_v4_horaria_anual.csv", parse_dates=["ini"], low_memory=False).assign(mc=m)
